# CS4200

# Assignment for Branch Prediction

Total 200 points:

* Predictor Implementation:
  * 15 points for completed implementation of the following three methods:
  * `next_predict`
  * `incorrect_predict`
  * `correct_predict`
* Simulations:
  * 25 points each for the following scenarios per predictor:
  * `next_branch_outcome_random`
  * `next_branch_outcome_loop`
  * `next_branch_outcome_long_loop`
* 10 points each for a discussion on the peformance of the predictor in the specified scenarios.

This assignment models the one-bit and two-bit branch predictors.
To simulate instructions and whether branches will occur or not, the provided methods `next_branch_outcome_random` and `next_branch_outcome_loop` will be used. These methods will simulate a completely random prediction outcome, and a set of outcomes that would more closely resemble a series of loops. You will then implement a method `next_branch_outcome_long_loop` which models a long unrolled loop. A return of `True` represents taking a branch, and a `False` represents not taking a branch.

The class `Predictor` represents the predictor. It is best practice to set the initial state to 0.

In [10]:
from random import paretovariate
from random import random

def next_branch_outcome_loop():
    alpha = 2
    outcome = paretovariate(alpha)
    outcome = outcome > 2
    return outcome

def next_branch_outcome_random():
    outcome = random()
    outcome = outcome > 0.5
    return outcome

def next_branch_outcome_long_loop():
    return random() < 0.9

def run_simulation(predictor, outcome_function, iterations=10000):
    """
    This function takes two positional arguments
    predictor is the class object for the predictor to be evaluated
    outcome_function is a function to run
    interations is optional but defaults to 10000
    This function returns the number of correct predictions / total iterations
    """
    correct = 0

    for _ in range(iterations):
        actual = outcome_function()
        prediction = predictor.next_predict()

        if prediction == actual:
            correct += 1
            predictor.correct_predict()
        else:
            predictor.incorrect_predict()

    return correct / iterations

class Predictor:
    
    def __init__(self):
        self.state = 0
    
    def next_predict(self):
        """
        Use this method to return the prediction based off of the current
        state.
        """
        raise NotImplementedError("Implement this method")
        
    def incorrect_predict(self):
        """
        Use this method to set the next state if an incorrect predict
        occurred. (self.state = next_state)
        """
        raise NotImplementedError("Implement this method")
        
    def correct_predict(self):
        """
        Use this method to set the next state if an incorrect predict
        occurred. (self.state = next_state)
        """
        raise NotImplementedError("Implement this method")

## One Bit Predictor

A one-bit predictor has two unique states to predict a branch to be taken or not taken.

```mermaid
flowchart LR
T((T)) --> |0| NT((NT))
NT --> |1| T
T --> |1| T
NT --> |0| NT
```

Complete the `OneBitPredictor` class by implementing the `next_predict`, `incorrect_predict`, and `correct_predict` methods. This instantiation will be used to compute the prediction accuracy. Use the `next_predict` method of the class to predict the next branch state. If the predict is incorrect, use the `incorrect_predict` method to set the next state. If the predict is correct, use the `correct_predict` method to set the next state.

In [5]:
class OneBitPredictor(Predictor):
    
    def next_predict(self):
        return self.state == 1
        
    def incorrect_predict(self):
        self.state = 1 - self.state
        
    def correct_predict(self):
        pass

### Random Branch Prediction

Use the `next_branch_outcome_random` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [16]:
one_bit_outcome_random = run_simulation(OneBitPredictor(), next_branch_outcome_random)
one_bit_outcome_random

0.4873

### Loop Branch Prediction

Use the `next_branch_outcome_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [19]:
one_bit_outcome_loop = run_simulation(OneBitPredictor(), next_branch_outcome_loop)
one_bit_outcome_loop

0.6272

### Long Loop Branch Prediction

Use the `next_branch_outcome_long_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [20]:
one_bit_outcome_long_loop = run_simulation(OneBitPredictor(), next_branch_outcome_long_loop)
one_bit_outcome_long_loop

0.8213

### Discussion

The one bit random branch prediction was about half correct. This makes sense because it would output true or false about half the time. If I ran more iterations it would probably get closer to 0.5. For the loop branch prediction it got around 0.62 because the output is usually over two but sometimes under. For the long loop it got 0.82 because the chance of returning true is around 90%. 

## Two Bit Predictor

A two-bit predictor has four unique states to predict a branch to be taken or not taken.

```mermaid
graph LR
    subgraph Taken
        WT((WT))
        T((T))
    end
    subgraph Not_Taken
        WNT((WNT))
        NT((NT))
    end
WT --> |0| Not_Taken
WNT --> |1| Taken
WNT --> |0| NT
NT --> |1| WNT
NT --> |0| NT
WT --> |1| T
T --> |0| WT
T --> |1| T
```

Complete the `TwoBitPredictor` class by implementing the `next_predict`, `incorrect_predict`, and `correct_predict` methods. This instantiation will be used to compute the prediction accuracy. Use the `next_predict` method of the class to predict the next branch state. If the predict is incorrect, use the `incorrect_predict` method to set the next state. If the predict is correct, use the `correct_predict` method to set the next state.

In [6]:
class TwoBitPredictor(Predictor):
    
    def next_predict(self):
        return self.state >= 2
        
    def incorrect_predict(self):
        """
        If the prediction was wrong take a step in the other direction
        """
        if self.state >= 2:
            self.state = max(0, self.state - 1)
        
    def correct_predict(self):
        """
        If the correct branch was taken increase the steps in that branch
        Else strengthen the other branch
        """
        if self.state >= 2:
            self.state = min(3, self.state + 1)
        else:
            self.state = max(0, self.state - 1)

### Random Branch Prediction

Use the `next_branch_outcome_random` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [21]:
two_bit_outcome_random = run_simulation(TwoBitPredictor(), next_branch_outcome_random)
two_bit_outcome_random

0.5004

### Loop Branch Prediction

Use the `next_branch_outcome_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [22]:
two_bit_outcome_loop = run_simulation(TwoBitPredictor(), next_branch_outcome_loop)
two_bit_outcome_loop

0.7493

### Long Loop Branch Prediction

Use the `next_branch_outcome_long_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (25 points)

In [23]:
two_bit_outcome_long_loop = run_simulation(TwoBitPredictor(), next_branch_outcome_long_loop)
two_bit_outcome_long_loop

0.1005

### Discussion

Two bit got closer to 50% on the random because the outcome is a 50% probability. For the loop branch it got around 74% because it takes longer for it to make a decision since it has two bits. For the long loop it performed poorly scoring 10% correct predictions. The long loophas a 90% chance of true which means it has sparse feedback on changes in direction. This also means it will take longer to get to change directions since it is a two bit.

## N-Bit Predictor

Extra credit: 75 points.
Inherit the `Predictor` class and implement it's methods just like before. Now, implement an `n-bit` predictor that represents a higher confidence prediction.

In [7]:
class NBitPredictor(Predictor):
    def __init__(self, n):
        super().__init__()
        self.n = n
        self.max_state = (2 ** n) - 1
        self.threshold = 2 ** (n - 1)
        self.state = 0

    def next_predict(self):
        """
        If the state taken is about the threshold it will return true
        Threshold should be the mid point of n, 2^(n-1)
        """
        return self.state >= self.threshold

    def incorrect_predict(self):
        """
        If we predicted incorrectly on taken then we decrement the state
        Else we predicted incorrectly on not taken we choose the higher stae or max_state
        Max state should be (2^n) - 1
        """
        if self.state >= self.threshold:
            self.state = max(0, self.state - 1)
        else:
            self.state = min(self.max_state, self.state + 1)

    def correct_predict(self):
        """
        If we predicted correct on prediction then increment
        If we predicted correct and wasnt taken decrement
        """
        if self.state >= self.threshold:
            self.state = min(self.max_state, self.state + 1)
        else:
            self.state = max(0, self.state - 1)

### Random Branch Prediction

Use the `next_branch_outcome_random` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (20 points)

In [25]:
n_bit_outcome_random = run_simulation(NBitPredictor(4), next_branch_outcome_random)
n_bit_outcome_random

0.5048

### Loop Branch Prediction

Use the `next_branch_outcome_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (20 points)

In [26]:
n_bit_outcome_loop = run_simulation(NBitPredictor(4), next_branch_outcome_loop)
n_bit_outcome_loop

0.7472

### Long Loop Branch Prediction

Use the `next_branch_outcome_long_loop` method to generate branch outcomes. Use the previously implemented methods to compute a prediction rate. (20 points)

In [27]:
n_bit_outcome_long_loop = run_simulation(NBitPredictor(4), next_branch_outcome_long_loop)
n_bit_outcome_long_loop

0.8998

### Discussion

For n bit prediction it acts almost like a 2 bit predictor as the threshold is the midpoint of the number of bits. This could also return odd results if the number of bits is odd since the mid point wont be in the middle. The random branch scored at almost exactly 50% which aligns with the trend I saw for the 1 and two bit predictors. The loop branch prediction showed 74% accuracy on predictions because it took longer to change branched. The one bit was able to correct itself faster. For the long loop it scored almost exactly at 90% and aligned almost perfectly with the actual outcome. This is because it acts like a two bit except it can get feedback faster and change directions like the one bit.